# E4a mailbox debug view (read-only observer)

Shows what the RV32I publishes in the shared BRAM mailbox, through **PYNQ `Overlay(download=False)`**
(hardware metadata from the HWH) and **`pynq.MMIO`** (`ps_mailbox.overlay_device.PynqMailboxDevice`).

* Load the E4a bitstream with the board tools first (`scripts/fpga/kv260_ps_board.py load`). This
  notebook never downloads a bitstream, never calls `Overlay(bitfile)` with its default download,
  never touches FPGA manager / device tree overlays / clocks.
* It is an **observer**: it reads the header and the newest record and never writes the consumer
  words, so it never competes with the reader (`python -m ps_mailbox.cli`) for the ring.
* `pynq.MMIO` maps `/dev/mem` and needs root; if this Jupyter kernel is not root, use the CLI instead
  (the Jupyter service is not to be switched to root for this).
* Stop this notebook (last cell) before the bitstream is unloaded.

In [ ]:
import sys, time
from pathlib import Path
BOARD_DIR = Path.home() / 'riscv-emg-soc' / 'e4a_b'   # scripts/fpga/kv260_e4b.py stage
PROG = 'mbx_rt2k_long'
sys.path.insert(0, str(BOARD_DIR))
from ps_mailbox import protocol as P
from ps_mailbox.overlay_device import PynqMailboxDevice
from ps_mailbox.pynq_reader import MailboxReader
dev = PynqMailboxDevice(BOARD_DIR / 'overlays' / PROG / f'{PROG}.bit', loaded_bin=BOARD_DIR / 'fw' / 'rv32_6a.bit.bin')
rep = dev.verify()            # artifacts, FPGA manager / overlay, Overlay(download=False) metadata, debug ID
ov = rep['overlay']
print('mailbox', ov['mailbox'], ' debug', ov['debug'])
print('PYNQ is_loaded() =', ov['is_loaded'], '(PYNQ 3.0.1, download=False: a file-name comparison only)')
print('ip_dict', ov['ip_dict_keys'])
print('mem_dict', ov['mem_dict_keys'])

In [ ]:
w = dev.open_mailbox()
print({k: hex(v) for k, v in dev.probe_header(w).items()})
obs = MailboxReader(w)        # used for read_header() / read_record() only: no CONSUMER_* writes
h = obs.read_header()
for k in ('epoch', 'state', 'producer_index', 'consumer_index', 'consumer_epoch', 'published_cnt', 'drop_cnt', 'error_cnt'):
    print(f'{k:16s} {h[k]}')

In [ ]:
t_end, last, t_last = time.monotonic() + 10, None, None
while time.monotonic() < t_end:
    h = obs.read_header()
    prod = h['producer_index']
    if prod != last and P.used(prod, h['epoch_start_index']) > 0:
        obs.epoch = h['epoch']
        r = obs.read_record((prod - 1) & P.MASK32)       # newest record, not acknowledged
        now = time.monotonic()
        if r is not None:
            rate = f'{1 / (now - t_last):5.1f} Hz' if t_last else '   -    '
            print(f"epoch {h['epoch']} #{r['result_index']:3d} {P.gesture_name(r['gesture']):11s} argmax {r['argmax']} "
                  f"scores {r['scores']} {rate} drop {h['drop_cnt']} err {h['error_cnt']} {P.STATE_NAMES[h['state']]}")
        last, t_last = prod, now
    time.sleep(0.01)

In [ ]:
rep = dev.close()             # unmap before the bitstream is unloaded
print('/dev/mem mappings left in this kernel:', rep['devmem_mappings_after_close'])